In [87]:
import pandas as pd
import glob
import os

In [88]:
# Read and concatenate all CSV files from woreda_crop_cover_change directory
csv_pattern = os.path.join('..', 'data', 'ethiopia', 'woreda_crop_cover_change', 'woreda_crop_batch_*.csv')
csv_files = sorted(glob.glob(csv_pattern))

print(f"Found {len(csv_files)} CSV files")

# Read and concatenate all CSV files
df_list = []
for csv_file in csv_files:
    df = pd.read_csv(csv_file)
    df_list.append(df)
    print(f"Read {os.path.basename(csv_file)}: {len(df)} rows")

# Concatenate all dataframes
crop_change_df = pd.concat(df_list, ignore_index=True)

print(f"\nTotal rows in concatenated dataframe: {len(crop_change_df)}")
print(f"Shape: {crop_change_df.shape}")
crop_change_df.head()

Found 23 CSV files
Read woreda_crop_batch_0.csv: 300 rows
Read woreda_crop_batch_1.csv: 300 rows
Read woreda_crop_batch_10.csv: 300 rows
Read woreda_crop_batch_11.csv: 300 rows
Read woreda_crop_batch_12.csv: 300 rows
Read woreda_crop_batch_13.csv: 300 rows
Read woreda_crop_batch_14.csv: 300 rows
Read woreda_crop_batch_15.csv: 300 rows
Read woreda_crop_batch_16.csv: 300 rows
Read woreda_crop_batch_17.csv: 300 rows
Read woreda_crop_batch_18.csv: 300 rows
Read woreda_crop_batch_19.csv: 300 rows
Read woreda_crop_batch_2.csv: 300 rows
Read woreda_crop_batch_20.csv: 300 rows
Read woreda_crop_batch_21.csv: 300 rows
Read woreda_crop_batch_22.csv: 260 rows
Read woreda_crop_batch_3.csv: 300 rows
Read woreda_crop_batch_4.csv: 300 rows
Read woreda_crop_batch_5.csv: 300 rows
Read woreda_crop_batch_6.csv: 300 rows
Read woreda_crop_batch_7.csv: 300 rows
Read woreda_crop_batch_8.csv: 300 rows
Read woreda_crop_batch_9.csv: 300 rows

Total rows in concatenated dataframe: 6860
Shape: (6860, 3)


,woreda,year,area_ha
0,AddisKetema,2015,0.297176
1,AddisKetema,2016,1.537608
2,AddisKetema,2017,3.120235
3,AddisKetema,2018,2.213216
4,AddisKetema,2019,0.837176


In [89]:
# Drop all rows where year=2015
crop_change_df = crop_change_df[crop_change_df['year'] != 2015]

print(f"Rows after dropping year 2015: {len(crop_change_df)}")
print(f"Shape: {crop_change_df.shape}")
crop_change_df.head()

Rows after dropping year 2015: 6174
Shape: (6174, 3)


,woreda,year,area_ha
1,AddisKetema,2016,1.537608
2,AddisKetema,2017,3.120235
3,AddisKetema,2018,2.213216
4,AddisKetema,2019,0.837176
5,AddisKetema,2020,0.000000


In [90]:
# Remove all rows for woredas that have any year with area_ha below 1000ha
# First, identify woredas with area_ha < 1000 in any year
woredas_below_threshold = crop_change_df[crop_change_df['area_ha'] < 1000]['woreda'].unique()

print(f"Found {len(woredas_below_threshold)} woredas with area_ha < 1000ha in at least one year")

# Remove all rows for those woredas
crop_change_df = crop_change_df[~crop_change_df['woreda'].isin(woredas_below_threshold)]

print(f"Rows after removing woredas: {len(crop_change_df)}")
print(f"Shape: {crop_change_df.shape}")

Found 65 woredas with area_ha < 1000ha in at least one year
Rows after removing woredas: 5589
Shape: (5589, 3)


In [91]:
# Create a new dataframe with crop cover change per woreda
# First, check for and handle duplicate woreda-year combinations by aggregating
crop_change_agg = crop_change_df.groupby(['woreda', 'year'], as_index=False)['area_ha'].sum()

# Pivot the data to have years as columns
pivot_df = crop_change_agg.pivot(index='woreda', columns='year', values='area_ha')

# Calculate year-over-year changes (both absolute and percent)
change_df = pd.DataFrame({'woreda': pivot_df.index})

# Add area_ha columns for each year
for year in range(2016, 2025):
    if year in pivot_df.columns:
        change_df[f'area_ha_{year}'] = pivot_df[year].values

# Add change columns
for year in range(2016, 2024):
    next_year = year + 1
    if year in pivot_df.columns and next_year in pivot_df.columns:
        year_suffix = f'{str(year)[2:]}_{str(next_year)[2:]}'
        
        # Absolute change
        change_df[f'change_{year_suffix}'] = pivot_df[next_year].values - pivot_df[year].values
        
        # Percent change
        change_df[f'percent_change_{year_suffix}'] = ((pivot_df[next_year].values - pivot_df[year].values) / pivot_df[year].values) * 100

print(f"Change dataframe shape: {change_df.shape}")
print(f"Columns: {list(change_df.columns)}")

Change dataframe shape: (607, 26)
Columns: ['woreda', 'area_ha_2016', 'area_ha_2017', 'area_ha_2018', 'area_ha_2019', 'area_ha_2020', 'area_ha_2021', 'area_ha_2022', 'area_ha_2023', 'area_ha_2024', 'change_16_17', 'percent_change_16_17', 'change_17_18', 'percent_change_17_18', 'change_18_19', 'percent_change_18_19', 'change_19_20', 'percent_change_19_20', 'change_20_21', 'percent_change_20_21', 'change_21_22', 'percent_change_21_22', 'change_22_23', 'percent_change_22_23', 'change_23_24', 'percent_change_23_24']


In [92]:
change_df

,woreda,area_ha_2016,area_ha_2017,area_ha_2018,area_ha_2019,area_ha_2020,area_ha_2021,area_ha_2022,area_ha_2023,area_ha_2024,...,change_19_20,percent_change_19_20,change_20_21,percent_change_20_21,change_21_22,percent_change_21_22,change_22_23,percent_change_22_23,change_23_24,percent_change_23_24
0,AbAla,3893.995647,5769.056118,3634.420667,2722.476706,4800.511451,4071.425608,2869.888118,2449.974275,2988.924588,...,2078.034745,76.328835,-729.085843,-15.187670,-1201.537490,-29.511469,-419.913843,-14.631715,538.950314,21.998203
1,Ababo,17193.028157,20064.302667,19454.348784,21440.230941,19125.850471,22330.603725,20055.097608,17820.037569,16047.600039,...,-2314.380471,-10.794569,3204.753255,16.756135,-2275.506118,-10.190079,-2235.060039,-11.144598,-1772.437529,-9.946318
2,AbayChomen,22291.330471,23768.482471,23232.468549,25193.342627,24085.553020,24299.573255,22500.448039,21906.957490,20609.058431,...,-1107.789608,-4.397152,214.020235,0.888583,-1799.125216,-7.403937,-593.490549,-2.637683,-1297.899059,-5.924598
3,Abaya,17704.290314,16055.569725,14254.189569,16927.478000,9228.087294,12158.729451,17780.714118,18176.419843,8785.148627,...,-7699.390706,-45.484571,2930.642157,31.757850,5621.984667,46.238258,395.705725,2.225477,-9391.271216,-51.667332
4,AbeDongoro,14038.284824,20677.359216,16295.743216,22732.226000,14594.987216,27842.919647,19861.127451,16769.212784,15127.873255,...,-8137.238784,-35.796049,13247.932431,90.770428,-7981.792196,-28.667224,-3091.914667,-15.567669,-1641.339529,-9.787815
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
602,YilmanaDensa,85091.139294,83157.676392,88745.133216,86608.151765,85446.375137,85729.698863,87165.031843,87160.558118,83943.195373,...,-1161.776627,-1.341417,283.323725,0.331581,1435.332980,1.674254,-4.473725,-0.005132,-3217.362745,-3.691306
603,Yubdo,2984.368000,3339.442196,5525.460941,4946.505882,4260.727569,4922.917922,4365.169686,3858.485961,3458.958314,...,-685.778314,-13.863894,662.190353,15.541720,-557.748235,-11.329627,-506.683725,-11.607423,-399.527647,-10.354519
604,Zala,9324.863490,20169.924902,16871.886039,26417.189843,17429.306745,21039.435216,23647.525882,29387.968902,22390.975686,...,-8987.883098,-34.022858,3610.128471,20.712978,2608.090667,12.396201,5740.443020,24.275026,-6996.993216,-23.809040
605,Ziquala,7625.451176,9256.904235,8795.758863,11125.286863,9878.042039,12848.619059,8900.939333,15558.254549,8368.634078,...,-1247.244824,-11.210900,2970.577020,30.072529,-3947.679725,-30.724545,6657.315216,74.793401,-7189.620471,-46.210971


In [93]:
conflict_df = pd.read_csv("../data/ethiopia/ethiopia_conflict_data.csv")
conflict_df

,id,relid,year,active_year,code_status,type_of_violence,conflict_dset_id,conflict_new_id,conflict_name,dyad_dset_id,...,best,high,low,gwnoa,gwnob,geometry,code,region_right,zone,woreda
0,370840,ETH-2020-1-555-0,2020,True,Clear,1,267,267,Ethiopia: Government,555,...,200,200,200,530.0,NaN,POINT (36.973604 13.56387),ETH.11.3.2_1,Tigray,Mi'irabawi,Tsegede
1,371338,ETH-2020-1-555-19,2020,True,Clear,1,267,267,Ethiopia: Government,555,...,3,3,2,530.0,NaN,POINT (39.47528 13.49667),ETH.11.1.4_1,Tigray,Debubawi,Enderta
2,510463,ETH-2020-1-555-99,2020,True,Clear,1,267,267,Ethiopia: Government,555,...,19254,17329,19254,530.0,NaN,POINT (39.5 13.5),ETH.11.1.4_1,Tigray,Debubawi,Enderta
3,484978,ETH-2020-1-555-87,2020,True,Clear,1,267,267,Ethiopia: Government,555,...,1,1,0,530.0,NaN,POINT (36.854864 13.980309),ETH.11.3.1_1,Tigray,Mi'irabawi,KaftaHumera
4,485544,ETH-2020-1-555-43.2,2020,True,Clear,1,267,267,Ethiopia: Government,555,...,253,254,253,530.0,NaN,POINT (39.726368 12.454523),ETH.11.1.7_1,Tigray,Debubawi,RayaAzebo
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3312,558584,ETH-2024-1-17147-308,2024,True,Clear,1,16069,16069,Ethiopia: Government/Amhara,17147,...,3,3,3,530.0,NaN,POINT (37.39077 11.59364),ETH.3.3.1_1,Amhara,BahirDarSpecialZone,BahirDar
3313,564230,ETH-2024-1-17147-313,2024,True,Clear,1,16069,16069,Ethiopia: Government/Amhara,17147,...,2,200,2,530.0,NaN,POINT (39.75 12.25),ETH.3.11.7_1,Amhara,SemenWello,Kobo
3314,386772,ETH-2021-3-17647-0,2021,True,Clear,3,8803,16023,OLA - Fekade Abdisa faction - Civilians,8803,...,29,29,29,NaN,NaN,POINT (37.159589 9.888141),ETH.8.6.7_1,Oromia,HoroGuduru,JarteJardega
3315,461355,ETH-2022-3-17647-0,2022,False,Clear,3,8803,16023,OLA - Fekade Abdisa faction - Civilians,8803,...,5,50,5,NaN,NaN,POINT (36.938963 10.143489),ETH.8.6.4_1,Oromia,HoroGuduru,Amuru


In [94]:
# Remove rows where the "best" column is above 1000
conflict_df = conflict_df[conflict_df['best'] <= 1000]

print(f"Rows after filtering: {len(conflict_df)}")
print(f"Shape: {conflict_df.shape}")
conflict_df.head()

Rows after filtering: 3310
Shape: (3310, 54)


,id,relid,year,active_year,code_status,type_of_violence,conflict_dset_id,conflict_new_id,conflict_name,dyad_dset_id,...,best,high,low,gwnoa,gwnob,geometry,code,region_right,zone,woreda
0,370840,ETH-2020-1-555-0,2020,True,Clear,1,267,267,Ethiopia: Government,555,...,200,200,200,530.0,NaN,POINT (36.973604 13.56387),ETH.11.3.2_1,Tigray,Mi'irabawi,Tsegede
1,371338,ETH-2020-1-555-19,2020,True,Clear,1,267,267,Ethiopia: Government,555,...,3,3,2,530.0,NaN,POINT (39.47528 13.49667),ETH.11.1.4_1,Tigray,Debubawi,Enderta
3,484978,ETH-2020-1-555-87,2020,True,Clear,1,267,267,Ethiopia: Government,555,...,1,1,0,530.0,NaN,POINT (36.854864 13.980309),ETH.11.3.1_1,Tigray,Mi'irabawi,KaftaHumera
4,485544,ETH-2020-1-555-43.2,2020,True,Clear,1,267,267,Ethiopia: Government,555,...,253,254,253,530.0,NaN,POINT (39.726368 12.454523),ETH.11.1.7_1,Tigray,Debubawi,RayaAzebo
5,485545,ETH-2020-1-555-43.3,2020,True,Clear,1,267,267,Ethiopia: Government,555,...,253,253,254,530.0,NaN,POINT (39.64479 12.797859),ETH.11.1.7_1,Tigray,Debubawi,RayaAzebo


In [95]:
conflict_df

,id,relid,year,active_year,code_status,type_of_violence,conflict_dset_id,conflict_new_id,conflict_name,dyad_dset_id,...,best,high,low,gwnoa,gwnob,geometry,code,region_right,zone,woreda
0,370840,ETH-2020-1-555-0,2020,True,Clear,1,267,267,Ethiopia: Government,555,...,200,200,200,530.0,NaN,POINT (36.973604 13.56387),ETH.11.3.2_1,Tigray,Mi'irabawi,Tsegede
1,371338,ETH-2020-1-555-19,2020,True,Clear,1,267,267,Ethiopia: Government,555,...,3,3,2,530.0,NaN,POINT (39.47528 13.49667),ETH.11.1.4_1,Tigray,Debubawi,Enderta
3,484978,ETH-2020-1-555-87,2020,True,Clear,1,267,267,Ethiopia: Government,555,...,1,1,0,530.0,NaN,POINT (36.854864 13.980309),ETH.11.3.1_1,Tigray,Mi'irabawi,KaftaHumera
4,485544,ETH-2020-1-555-43.2,2020,True,Clear,1,267,267,Ethiopia: Government,555,...,253,254,253,530.0,NaN,POINT (39.726368 12.454523),ETH.11.1.7_1,Tigray,Debubawi,RayaAzebo
5,485545,ETH-2020-1-555-43.3,2020,True,Clear,1,267,267,Ethiopia: Government,555,...,253,253,254,530.0,NaN,POINT (39.64479 12.797859),ETH.11.1.7_1,Tigray,Debubawi,RayaAzebo
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3312,558584,ETH-2024-1-17147-308,2024,True,Clear,1,16069,16069,Ethiopia: Government/Amhara,17147,...,3,3,3,530.0,NaN,POINT (37.39077 11.59364),ETH.3.3.1_1,Amhara,BahirDarSpecialZone,BahirDar
3313,564230,ETH-2024-1-17147-313,2024,True,Clear,1,16069,16069,Ethiopia: Government/Amhara,17147,...,2,200,2,530.0,NaN,POINT (39.75 12.25),ETH.3.11.7_1,Amhara,SemenWello,Kobo
3314,386772,ETH-2021-3-17647-0,2021,True,Clear,3,8803,16023,OLA - Fekade Abdisa faction - Civilians,8803,...,29,29,29,NaN,NaN,POINT (37.159589 9.888141),ETH.8.6.7_1,Oromia,HoroGuduru,JarteJardega
3315,461355,ETH-2022-3-17647-0,2022,False,Clear,3,8803,16023,OLA - Fekade Abdisa faction - Civilians,8803,...,5,50,5,NaN,NaN,POINT (36.938963 10.143489),ETH.8.6.4_1,Oromia,HoroGuduru,Amuru


In [96]:
# Sum "best" by year and woreda
conflict_agg = conflict_df.groupby(['woreda', 'year'], as_index=False)['best'].sum()

print(f"Aggregated conflict data shape: {conflict_agg.shape}")
conflict_agg.head(10)

Aggregated conflict data shape: (868, 3)


,woreda,year,best
0,AbAla,2020,0
1,AbAla,2021,286
2,AbAla,2022,24
3,Ababo,2022,0
4,Ababo,2023,8
5,Ababo,2024,2
6,AbayChomen,2021,0
7,AbayChomen,2022,0
8,AbayChomen,2023,20
9,AbayChomen,2024,17


In [97]:
# Transform to have estimated_deaths columns by year
conflict_pivot = conflict_agg.pivot(index='woreda', columns='year', values='best').reset_index()

# Rename columns to estimated_deaths_YYYY format
conflict_pivot.columns = ['woreda'] + [f'estimated_deaths_{int(year)}' if year != 'woreda' else year 
                                        for year in conflict_pivot.columns[1:]]

# Fill NaN values with 0 (woredas with no conflicts in certain years)
for year in range(2020, 2025):
    col_name = f'estimated_deaths_{year}'
    if col_name in conflict_pivot.columns:
        conflict_pivot[col_name] = conflict_pivot[col_name].fillna(0)

print(f"Conflict pivot shape: {conflict_pivot.shape}")
print(f"Columns: {list(conflict_pivot.columns)}")
conflict_pivot.head(10)

Conflict pivot shape: (375, 6)
Columns: ['woreda', 'estimated_deaths_2020', 'estimated_deaths_2021', 'estimated_deaths_2022', 'estimated_deaths_2023', 'estimated_deaths_2024']


,woreda,estimated_deaths_2020,estimated_deaths_2021,estimated_deaths_2022,estimated_deaths_2023,estimated_deaths_2024
0,AbAla,0.0,286.0,24.0,0.0,0.0
1,Ababo,0.0,0.0,0.0,8.0,2.0
2,AbayChomen,0.0,0.0,0.0,20.0,17.0
3,Abaya,0.0,0.0,1.0,0.0,1.0
4,AbeDongoro,0.0,0.0,0.0,12.0,45.0
5,Abergele,0.0,20.0,10.0,0.0,0.0
6,AbichunaGne'a,0.0,0.0,0.0,0.0,0.0
7,Abobo,0.0,0.0,0.0,2.0,1.0
8,AbunaG/Beret,0.0,1.0,62.0,3.0,11.0
9,Ada'a,0.0,0.0,2.0,4.0,0.0


In [98]:
# Join conflict_pivot to change_df by woreda
# Using left join to keep only woredas that exist in change_df
final_df = change_df.merge(conflict_pivot, on='woreda', how='left')

# Fill NaN values with 0 for woredas that had no conflicts
estimated_deaths_cols = [col for col in final_df.columns if col.startswith('estimated_deaths_')]
for col in estimated_deaths_cols:
    final_df[col] = final_df[col].fillna(0)

print(f"Final dataframe shape: {final_df.shape}")
print(f"Columns: {list(final_df.columns)}")
final_df.head()

Final dataframe shape: (607, 31)
Columns: ['woreda', 'area_ha_2016', 'area_ha_2017', 'area_ha_2018', 'area_ha_2019', 'area_ha_2020', 'area_ha_2021', 'area_ha_2022', 'area_ha_2023', 'area_ha_2024', 'change_16_17', 'percent_change_16_17', 'change_17_18', 'percent_change_17_18', 'change_18_19', 'percent_change_18_19', 'change_19_20', 'percent_change_19_20', 'change_20_21', 'percent_change_20_21', 'change_21_22', 'percent_change_21_22', 'change_22_23', 'percent_change_22_23', 'change_23_24', 'percent_change_23_24', 'estimated_deaths_2020', 'estimated_deaths_2021', 'estimated_deaths_2022', 'estimated_deaths_2023', 'estimated_deaths_2024']


,woreda,area_ha_2016,area_ha_2017,area_ha_2018,area_ha_2019,area_ha_2020,area_ha_2021,area_ha_2022,area_ha_2023,area_ha_2024,...,percent_change_21_22,change_22_23,percent_change_22_23,change_23_24,percent_change_23_24,estimated_deaths_2020,estimated_deaths_2021,estimated_deaths_2022,estimated_deaths_2023,estimated_deaths_2024
0,AbAla,3893.995647,5769.056118,3634.420667,2722.476706,4800.511451,4071.425608,2869.888118,2449.974275,2988.924588,...,-29.511469,-419.913843,-14.631715,538.950314,21.998203,0.0,286.0,24.0,0.0,0.0
1,Ababo,17193.028157,20064.302667,19454.348784,21440.230941,19125.850471,22330.603725,20055.097608,17820.037569,16047.600039,...,-10.190079,-2235.060039,-11.144598,-1772.437529,-9.946318,0.0,0.0,0.0,8.0,2.0
2,AbayChomen,22291.330471,23768.482471,23232.468549,25193.342627,24085.553020,24299.573255,22500.448039,21906.957490,20609.058431,...,-7.403937,-593.490549,-2.637683,-1297.899059,-5.924598,0.0,0.0,0.0,20.0,17.0
3,Abaya,17704.290314,16055.569725,14254.189569,16927.478000,9228.087294,12158.729451,17780.714118,18176.419843,8785.148627,...,46.238258,395.705725,2.225477,-9391.271216,-51.667332,0.0,0.0,1.0,0.0,1.0
4,AbeDongoro,14038.284824,20677.359216,16295.743216,22732.226000,14594.987216,27842.919647,19861.127451,16769.212784,15127.873255,...,-28.667224,-3091.914667,-15.567669,-1641.339529,-9.787815,0.0,0.0,0.0,12.0,45.0


In [99]:
conflict_agg

,woreda,year,best
0,AbAla,2020,0
1,AbAla,2021,286
2,AbAla,2022,24
3,Ababo,2022,0
4,Ababo,2023,8
...,...,...,...
863,ZiwayDugda,2020,59
864,ZiwayDugda,2021,4
865,ZiwayDugda,2022,9
866,ZiwayDugda,2023,39


In [100]:
# Transform to have estimated_deaths columns by year
conflict_pivot = conflict_agg.pivot(index='woreda', columns='year', values='best').reset_index()

# Rename columns to estimated_deaths_YYYY format
conflict_pivot.columns = ['woreda'] + [f'estimated_deaths_{int(year)}' if year != 'woreda' else year 
                                        for year in conflict_pivot.columns[1:]]

# Fill NaN values with 0 (woredas with no conflicts in certain years)
for year in range(2020, 2025):
    col_name = f'estimated_deaths_{year}'
    if col_name in conflict_pivot.columns:
        conflict_pivot[col_name] = conflict_pivot[col_name].fillna(0)

print(f"Conflict pivot shape: {conflict_pivot.shape}")
print(f"Columns: {list(conflict_pivot.columns)}")
conflict_pivot.head(10)

Conflict pivot shape: (375, 6)
Columns: ['woreda', 'estimated_deaths_2020', 'estimated_deaths_2021', 'estimated_deaths_2022', 'estimated_deaths_2023', 'estimated_deaths_2024']


,woreda,estimated_deaths_2020,estimated_deaths_2021,estimated_deaths_2022,estimated_deaths_2023,estimated_deaths_2024
0,AbAla,0.0,286.0,24.0,0.0,0.0
1,Ababo,0.0,0.0,0.0,8.0,2.0
2,AbayChomen,0.0,0.0,0.0,20.0,17.0
3,Abaya,0.0,0.0,1.0,0.0,1.0
4,AbeDongoro,0.0,0.0,0.0,12.0,45.0
5,Abergele,0.0,20.0,10.0,0.0,0.0
6,AbichunaGne'a,0.0,0.0,0.0,0.0,0.0
7,Abobo,0.0,0.0,0.0,2.0,1.0
8,AbunaG/Beret,0.0,1.0,62.0,3.0,11.0
9,Ada'a,0.0,0.0,2.0,4.0,0.0


In [101]:
conflict_pivot

,woreda,estimated_deaths_2020,estimated_deaths_2021,estimated_deaths_2022,estimated_deaths_2023,estimated_deaths_2024
0,AbAla,0.0,286.0,24.0,0.0,0.0
1,Ababo,0.0,0.0,0.0,8.0,2.0
2,AbayChomen,0.0,0.0,0.0,20.0,17.0
3,Abaya,0.0,0.0,1.0,0.0,1.0
4,AbeDongoro,0.0,0.0,0.0,12.0,45.0
...,...,...,...,...,...,...
370,YilmanaDensa,0.0,0.0,0.0,12.0,75.0
371,Yubdo,0.0,0.0,0.0,0.0,5.0
372,Zala,25.0,0.0,0.0,0.0,0.0
373,Ziquala,0.0,0.0,1.0,0.0,0.0


In [102]:
# Join conflict_pivot to change_df by woreda
# Using left join to keep only woredas that exist in change_df
final_df = change_df.merge(conflict_pivot, on='woreda', how='right')

# Fill NaN values with 0 for woredas that had no conflicts
estimated_deaths_cols = [col for col in final_df.columns if col.startswith('estimated_deaths_')]
for col in estimated_deaths_cols:
    final_df[col] = final_df[col].fillna(0)

print(f"Final dataframe shape: {final_df.shape}")
print(f"Columns: {list(final_df.columns)}")
final_df.head()

Final dataframe shape: (375, 31)
Columns: ['woreda', 'area_ha_2016', 'area_ha_2017', 'area_ha_2018', 'area_ha_2019', 'area_ha_2020', 'area_ha_2021', 'area_ha_2022', 'area_ha_2023', 'area_ha_2024', 'change_16_17', 'percent_change_16_17', 'change_17_18', 'percent_change_17_18', 'change_18_19', 'percent_change_18_19', 'change_19_20', 'percent_change_19_20', 'change_20_21', 'percent_change_20_21', 'change_21_22', 'percent_change_21_22', 'change_22_23', 'percent_change_22_23', 'change_23_24', 'percent_change_23_24', 'estimated_deaths_2020', 'estimated_deaths_2021', 'estimated_deaths_2022', 'estimated_deaths_2023', 'estimated_deaths_2024']


,woreda,area_ha_2016,area_ha_2017,area_ha_2018,area_ha_2019,area_ha_2020,area_ha_2021,area_ha_2022,area_ha_2023,area_ha_2024,...,percent_change_21_22,change_22_23,percent_change_22_23,change_23_24,percent_change_23_24,estimated_deaths_2020,estimated_deaths_2021,estimated_deaths_2022,estimated_deaths_2023,estimated_deaths_2024
0,AbAla,3893.995647,5769.056118,3634.420667,2722.476706,4800.511451,4071.425608,2869.888118,2449.974275,2988.924588,...,-29.511469,-419.913843,-14.631715,538.950314,21.998203,0.0,286.0,24.0,0.0,0.0
1,Ababo,17193.028157,20064.302667,19454.348784,21440.230941,19125.850471,22330.603725,20055.097608,17820.037569,16047.600039,...,-10.190079,-2235.060039,-11.144598,-1772.437529,-9.946318,0.0,0.0,0.0,8.0,2.0
2,AbayChomen,22291.330471,23768.482471,23232.468549,25193.342627,24085.553020,24299.573255,22500.448039,21906.957490,20609.058431,...,-7.403937,-593.490549,-2.637683,-1297.899059,-5.924598,0.0,0.0,0.0,20.0,17.0
3,Abaya,17704.290314,16055.569725,14254.189569,16927.478000,9228.087294,12158.729451,17780.714118,18176.419843,8785.148627,...,46.238258,395.705725,2.225477,-9391.271216,-51.667332,0.0,0.0,1.0,0.0,1.0
4,AbeDongoro,14038.284824,20677.359216,16295.743216,22732.226000,14594.987216,27842.919647,19861.127451,16769.212784,15127.873255,...,-28.667224,-3091.914667,-15.567669,-1641.339529,-9.787815,0.0,0.0,0.0,12.0,45.0


In [103]:
# Fill all NaN values with 0, then convert all columns except 'woreda' to integers
final_df = final_df.fillna(0)

for col in final_df.columns:
    if col != 'woreda':
        final_df[col] = final_df[col].astype(int)

print(f"Data types:\n{final_df.dtypes}")
final_df.head()

Data types:
woreda                   object
area_ha_2016              int64
area_ha_2017              int64
area_ha_2018              int64
area_ha_2019              int64
area_ha_2020              int64
area_ha_2021              int64
area_ha_2022              int64
area_ha_2023              int64
area_ha_2024              int64
change_16_17              int64
percent_change_16_17      int64
change_17_18              int64
percent_change_17_18      int64
change_18_19              int64
percent_change_18_19      int64
change_19_20              int64
percent_change_19_20      int64
change_20_21              int64
percent_change_20_21      int64
change_21_22              int64
percent_change_21_22      int64
change_22_23              int64
percent_change_22_23      int64
change_23_24              int64
percent_change_23_24      int64
estimated_deaths_2020     int64
estimated_deaths_2021     int64
estimated_deaths_2022     int64
estimated_deaths_2023     int64
estimated_deaths_2024     in

,woreda,area_ha_2016,area_ha_2017,area_ha_2018,area_ha_2019,area_ha_2020,area_ha_2021,area_ha_2022,area_ha_2023,area_ha_2024,...,percent_change_21_22,change_22_23,percent_change_22_23,change_23_24,percent_change_23_24,estimated_deaths_2020,estimated_deaths_2021,estimated_deaths_2022,estimated_deaths_2023,estimated_deaths_2024
0,AbAla,3893,5769,3634,2722,4800,4071,2869,2449,2988,...,-29,-419,-14,538,21,0,286,24,0,0
1,Ababo,17193,20064,19454,21440,19125,22330,20055,17820,16047,...,-10,-2235,-11,-1772,-9,0,0,0,8,2
2,AbayChomen,22291,23768,23232,25193,24085,24299,22500,21906,20609,...,-7,-593,-2,-1297,-5,0,0,0,20,17
3,Abaya,17704,16055,14254,16927,9228,12158,17780,18176,8785,...,46,395,2,-9391,-51,0,0,1,0,1
4,AbeDongoro,14038,20677,16295,22732,14594,27842,19861,16769,15127,...,-28,-3091,-15,-1641,-9,0,0,0,12,45


In [104]:
# Drop all rows where area_ha_2016 is 0
final_df = final_df[final_df['area_ha_2016'] != 0]

print(f"Rows after dropping area_ha_2016 = 0: {len(final_df)}")
print(f"Shape: {final_df.shape}")
final_df.head()

Rows after dropping area_ha_2016 = 0: 356
Shape: (356, 31)


,woreda,area_ha_2016,area_ha_2017,area_ha_2018,area_ha_2019,area_ha_2020,area_ha_2021,area_ha_2022,area_ha_2023,area_ha_2024,...,percent_change_21_22,change_22_23,percent_change_22_23,change_23_24,percent_change_23_24,estimated_deaths_2020,estimated_deaths_2021,estimated_deaths_2022,estimated_deaths_2023,estimated_deaths_2024
0,AbAla,3893,5769,3634,2722,4800,4071,2869,2449,2988,...,-29,-419,-14,538,21,0,286,24,0,0
1,Ababo,17193,20064,19454,21440,19125,22330,20055,17820,16047,...,-10,-2235,-11,-1772,-9,0,0,0,8,2
2,AbayChomen,22291,23768,23232,25193,24085,24299,22500,21906,20609,...,-7,-593,-2,-1297,-5,0,0,0,20,17
3,Abaya,17704,16055,14254,16927,9228,12158,17780,18176,8785,...,46,395,2,-9391,-51,0,0,1,0,1
4,AbeDongoro,14038,20677,16295,22732,14594,27842,19861,16769,15127,...,-28,-3091,-15,-1641,-9,0,0,0,12,45


In [105]:
final_df

,woreda,area_ha_2016,area_ha_2017,area_ha_2018,area_ha_2019,area_ha_2020,area_ha_2021,area_ha_2022,area_ha_2023,area_ha_2024,...,percent_change_21_22,change_22_23,percent_change_22_23,change_23_24,percent_change_23_24,estimated_deaths_2020,estimated_deaths_2021,estimated_deaths_2022,estimated_deaths_2023,estimated_deaths_2024
0,AbAla,3893,5769,3634,2722,4800,4071,2869,2449,2988,...,-29,-419,-14,538,21,0,286,24,0,0
1,Ababo,17193,20064,19454,21440,19125,22330,20055,17820,16047,...,-10,-2235,-11,-1772,-9,0,0,0,8,2
2,AbayChomen,22291,23768,23232,25193,24085,24299,22500,21906,20609,...,-7,-593,-2,-1297,-5,0,0,0,20,17
3,Abaya,17704,16055,14254,16927,9228,12158,17780,18176,8785,...,46,395,2,-9391,-51,0,0,1,0,1
4,AbeDongoro,14038,20677,16295,22732,14594,27842,19861,16769,15127,...,-28,-3091,-15,-1641,-9,0,0,0,12,45
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
370,YilmanaDensa,85091,83157,88745,86608,85446,85729,87165,87160,83943,...,1,-4,0,-3217,-3,0,0,0,12,75
371,Yubdo,2984,3339,5525,4946,4260,4922,4365,3858,3458,...,-11,-506,-11,-399,-10,0,0,0,0,5
372,Zala,9324,20169,16871,26417,17429,21039,23647,29387,22390,...,12,5740,24,-6996,-23,25,0,0,0,0
373,Ziquala,7625,9256,8795,11125,9878,12848,8900,15558,8368,...,-30,6657,74,-7189,-46,0,0,1,0,0


In [106]:
final_df.to_csv("../data/ethiopia/output/ethiopia_crop_and_conflict.csv")